In [14]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import chromadb
import json
import scrapy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from urllib.parse import urljoin

In [25]:
#Replace with nhs stuff and webscraper later

Base_URL_REQUEST = requests.get("https://www.nhs.uk/symptoms/")
BASE_URL = 'https://www.nhs.uk/symptoms/'
#Scrape nhs website

soup = BeautifulSoup(Base_URL_REQUEST.content,'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="Find out about symptoms such as pain, stomach problems or skin symptoms, including causes, treatment and what to do." name="description"/>
  <meta content="a4yrvgi5ZlBnKWfqFKkQ3_mEjqow_fpwbtF2bUTmZgc" name="google-site-verification"/>
  <link href="https://www.nhs.uk/symptoms/" rel="canonical"/>
  <title>
   Symptoms A to Z - NHS
  </title>
  <link crossorigin="" href="https://assets.nhs.uk/" rel="preconnect"/>
  <link as="font" crossorigin="" href="https://assets.nhs.uk/fonts/FrutigerLTW01-55Roman.woff2" rel="preload" type="font/woff2"/>
  <link as="font" crossorigin="" href="https://assets.nhs.uk/fonts/FrutigerLTW01-65Bold.woff2" rel="preload" type="font/woff2"/>
  <link href="/static/nhsuk/css/main.6a2df81b6189.css" rel="stylesheet" type="text/css"/>
  <sc

In [26]:
print(Base_URL_REQUEST.status_code)

200


In [30]:
symptom_links = {}
seen_urls = set()


for item in soup.select('ul.nhsuk-list.nhsuk-list--border.nhsuk-list--links'):
    

    for link in item.select('li a[href]'):
        
        url = urljoin(BASE_URL, link['href'])
        term = link.get_text('',strip=True)
        
        if url not in seen_urls:
            symptom_links[term] = url
            seen_urls.add(url)
            
print(symptom_links)

{'Acid reflux in babies, see Reflux in babies': 'https://www.nhs.uk/conditions/reflux-in-babies/', 'Acid reflux, see Heartburn and acid reflux': 'https://www.nhs.uk/conditions/heartburn-and-acid-reflux/', 'Amnesia, see Memory loss (amnesia)': 'https://www.nhs.uk/symptoms/memory-loss-amnesia/', 'Anal pain': 'https://www.nhs.uk/symptoms/anal-pain/', 'Anger': 'https://www.nhs.uk/mental-health/feelings-symptoms-behaviours/feelings-and-symptoms/anger/', 'Ankle pain': 'https://www.nhs.uk/symptoms/foot-pain/ankle-pain/', 'Ankles (swollen), see Swollen ankles, feet and legs (oedema)': 'https://www.nhs.uk/conditions/oedema/', 'Anosmia, see Lost or changed sense of smell': 'https://www.nhs.uk/symptoms/lost-or-changed-sense-smell/', 'Anus (itchy), see Itchy bottom': 'https://www.nhs.uk/symptoms/itchy-anus/', 'Anxiety, fear and panic': 'https://www.nhs.uk/mental-health/feelings-symptoms-behaviours/feelings-and-symptoms/anxiety-fear-panic/', 'Arm pain, see Elbow and arm pain': 'https://www.nhs.uk/s

In [43]:
def scrape_symptom_page(term,url):
    
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    page = BeautifulSoup(response.content, 'html.parser')
    content = page.find('article') or page
    
    records = []
    heading = 'introduction'
    text_parts = []
    
    def save_sections():
        
        text = "\n".join(text_parts).strip()
        if text:
            
            records.append({
                
                'term': term,
                'heading': heading,
                'text': text,
                'url': url
            }
            )
        
        
    for element in content.select('h2,h3, p, li'):
        
        if element.name in ('h2,h3'):
            save_sections()
            heading = element.get_text('',strip=True)
            text_parts = []
        else:
            
            if element.name == 'p' and element.find_parent('li'):
                continue
            
            text = element.get_text(' ', strip=True)
            
            if text:
                
                text = element.get_text(' ', strip=True)
                if text:
                    text_parts.append(text)
                    
                    
    save_sections()
    return records  
    


In [44]:
term = "Headaches"
url = symptom_links[term]

headache_sections = scrape_symptom_page(term, url)

for section in headache_sections[:3]:
    print(section["heading"])
    print(section["text"][:300])
    print(section["url"])
    print()

How you can ease headaches yourself
Headaches can last 30 minutes, several hours, or sometimes several days.
https://www.nhs.uk/symptoms/headaches/

Do
drink plenty of water
get plenty of rest if you also have a cold or the flu
try to relax – stress can make headaches worse
take paracetamol or ibuprofen
try to stay at home and avoid contact with other people if you also have a high temperature or you do not feel well enough to do your normal activi
https://www.nhs.uk/symptoms/headaches/

Don’t
do not skip meals (even if you might not feel like eating anything)
do not sleep more than you usually would – it can make the headache worse
do not strain your eyes for a long time – for example, by looking at a screen
do not drink alcohol
https://www.nhs.uk/symptoms/headaches/



In [ ]:
records = []

for term, url in list(symptom_links.items()):
    records.extend(scrape_symptom_page(term,url))

In [49]:
url = symptom_links["Headaches"]

page_records = [
    record for record in records
    if record["url"] == url
]

print("Sections on this page:", len(page_records))
for record in page_records:
    print("\n###", record["heading"])
    print(record["text"])

Sections on this page: 8

### How you can ease headaches yourself
Headaches can last 30 minutes, several hours, or sometimes several days.

### Do
drink plenty of water
get plenty of rest if you also have a cold or the flu
try to relax – stress can make headaches worse
take paracetamol or ibuprofen
try to stay at home and avoid contact with other people if you also have a high temperature or you do not feel well enough to do your normal activities

### Don’t
do not skip meals (even if you might not feel like eating anything)
do not sleep more than you usually would – it can make the headache worse
do not strain your eyes for a long time – for example, by looking at a screen
do not drink alcohol

### When and where to get medical help
Most headaches are not serious but sometimes you may need medical help.

### Non-urgent advice:See a GP if:
you've tried treating a headache yourself and it's not getting better or is getting worse
you regularly get headaches

### Urgent advice:Ask for an 

In [50]:
documents = [record['text'] for record in records]



In [51]:
embedding_model = SentenceTransformer('all-miniLM-L6-v2')

doc_embeddings = embedding_model.encode(
    
    documents,
    normalize_embeddings = True
).astype("float32")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 785.07it/s]


In [52]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

In [53]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)



Loading weights: 100%|██████████| 282/282 [00:01<00:00, 200.73it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [59]:
def rag_answer(query, top_k=2, threshold=0.3):
    
    query_embedding = embedding_model.encode(
        
        [query],
        normalize_embeddings=True
        
    ).astype("float32")
    
    distances, indices = index.search(query_embedding, top_k)

    if distances[0][0] < threshold:
        
        return None, "Sorry I could not find a relevant document to answer your question."
    
    retrieved_docs = [
        
        documents[i]
        for i in indices[0]
        if i != -1
    ]
    
    context = "\n".join(retrieved_docs)
    
    prompt = f"""
    
    Answer the question using only the information in the context.
    
    Context:
    {context}
    
    Question:
    {query}
    
    Answer:
    """
    
    inputs = tokenizer(
        
        prompt,
        return_tensors="pt",
        truncation=True
        
    )
    
    outputs = model.generate(
        
        **inputs,
        max_new_tokens = 50,
        do_sample=False,
        num_beams=4
    )
    
    response = tokenizer.decode(
        
        outputs[0],
        skip_special_tokens=True
        
    ).strip()
    
    
    return context,response

In [60]:
def run_qa_bot():
    
    print('Welcome to the RAG Q&A Bot! Ask a question or type "exit" to quit.')
    
    while True:
        query = input("User: ")
        
        #Maybe include escape key too?
        if query.lower() == "exit":
            
            print("Goodbye!")
            
            break
        
        if not query.strip():
            print("Please enter a valid question")
            continue
        
        context, answer = rag_answer(query)
        
        print(f"HelpBot: {context}\n")
        print(f"HelpBot: {answer}\n")
        
        
        
run_qa_bot()

Welcome to the RAG Q&A Bot! Ask a question or type "exit" to quit.
HelpBot: You or a child has a headache and:
has had a seizure (fit)
has numbness or weakness in the body or face
it started suddenly and is extremely painful
has had a head injury within the last 3 months – for example, from a fall or accident
are finding it difficult to speak, balance, walk or remember things
are drowsy or confused
has loss of vision
has a rash that does not fade when a glass is rolled over it
has a very high temperature, a stiff neck, or bright lights are bothering you
Do not drive to A&E. Ask someone to drive you or call 999 and ask for an ambulance.
Bring any medicines you take with you.
Eaaw https://www.shutterstock.com/image-photo/seborrheic-dermatitis-on-head-asian-baby-561450823
Yellow or white, greasy, scaly patches on your baby's scalp could be cradle cap.
Cradle cap can usually be treated at home.

HelpBot: a rash that does not fade when a glass is rolled over it has a very high temperature, 